In [1]:
#part a

In [2]:
import random

demands = [12,45,23,67,34,19, 56,38,72,15,49,61,
    27,83,41,55,30,77, 64,18,52,39,71,26,
    44,91,33,58,22,85, 16,69,47,74,31,53]

supply_penalty = 5
num_drivers = 10

In [3]:
def fitness(state, demands):

    total_demand = sum(demands[i] for i in state)
    penalty = supply_penalty * num_drivers
    return total_demand - penalty

#computes the objective by summing the demand scores of the 10
#driver zones and subtracting the total supply penalty

In [11]:
def random_state():

    return list(random.sample(range(36), num_drivers))

In [19]:
def get_neighbours(state):

    neighbours = []
    placed = list(state)
    unplaced = [i for i in range(36) if i not in state]

    for i in placed:
        for j in unplaced:
            neighbour = set(state)
            neighbour.remove(i)
            neighbour.add(j)
            neighbours.append(neighbour)
    return neighbours

#each neighbour swaps one driver from a placed zone to any
#unoccupied zone. returns a list of new sets, all of size 10,
#no duplicates.

In [20]:
for i in range(3):
    state = random_state()
    print("\nstate:", state)
    print("fitness:", fitness(state, demands))
    print("number of neighbours:", len(get_neighbours(state)))


state: [32, 24, 12, 33, 31, 6, 10, 0, 34, 14]
fitness: 400
number of neighbours: 260

state: [1, 26, 14, 5, 24, 28, 16, 2, 25, 4]
fitness: 332
number of neighbours: 260

state: [15, 19, 4, 16, 11, 34, 29, 21, 24, 8]
fitness: 419
number of neighbours: 260


In [13]:
#part b

In [14]:
def hc_driver(state, demands, variant='first_choice'):

    current = set(state)
    current_fitness = fitness(current, demands) #fitness function
    steps = 0

    while True:
        steps += 1
        neighbours = get_neighbours(current) #keep climbing until local max

        if variant == 'first_choice':
            moved = False
            for n in neighbours:
                n_fitness = fitness(n, demands) #generate neighbours
                if n_fitness > current_fitness:
                    current, current_fitness = n, n_fitness
                    moved = True
                    break  #first improvement
            if not moved:
                break  #local maximum

        elif variant == 'stochastic':
            better_neighbours = [n for n in neighbours if fitness(n, demands) > current_fitness]
            if not better_neighbours:
                break
            current = random.choice(better_neighbours)
            current_fitness = fitness(current, demands) #same work done for stochastic


    return current, current_fitness, steps

In [118]:
def rrhc_driver(num_restarts, demands, variant='first_choice'):
    best_overall = None #best state found from all restarts
    best_fitness_overall = -float('inf')
    per_restart_best = [] #list to store best fitness from all restarts

    for i in range(num_restarts):
        start_state = random_state()
        final_state, final_fitness, _ = hc_driver(start_state, demands, variant)

        per_restart_best.append(final_fitness)

        if final_fitness > best_fitness_overall:
            best_fitness_overall = final_fitness
            best_overall = final_state

    return best_overall, best_fitness_overall, per_restart_best

#we run hill climbing from multiple random starting states
#then track the best solution from each restart
#we keep the overall best solution across all restarts
#finally, return the best state, its fitness, and per-restart best

I chose First-Choice Hill Climbing because the neighbourhood size is large (260 neighbours) Scanning all neighbours every step can be slow, but First-Choice stops at the first improvement. Stochastic HC could explore more randomly but would take longer in such a large neighbourhood.

In [78]:
best_state, best_fit, per_restart_best = rrhc_driver(num_restarts=30, demands=demands, variant='first_choice')

print("best state found:", best_state)
print("fitness:", best_fit)
print("driver zones (row, col):", [(i//6, i%6) for i in best_state])
print("best fitness per restart:", per_restart_best)

best state found: {33, 3, 8, 13, 17, 18, 22, 25, 29, 31}
fitness: 703
driver zones (row, col): [(5, 3), (0, 3), (1, 2), (2, 1), (2, 5), (3, 0), (3, 4), (4, 1), (4, 5), (5, 1)]
best fitness per restart: [703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703]


In [22]:
#part c

In [23]:
def ga_fitness(chromosome, demands):
    #chromosome - list of 10 unique zone indices
    return sum(demands[i] for i in chromosome) - 5*10

In [93]:
def ordered_crossover(p1, p2):
    size = len(p1)

    #select two crossover points
    start, end = sorted(random.sample(range(size), 2))

    #initialize child with None values
    child = [None] * size

    #copy slice from parent1
    child[start:end+1] = p1[start:end+1]

    #fill remaining positions from parent2 in order
    p2_index = 0
    for i in range(size):
        if child[i] is None:
            #skip values already in child
            while p2[p2_index] in child:
                p2_index += 1
            child[i] = p2[p2_index]
            p2_index += 1

    return child

In [94]:
def ga_mutate(chromosome, pm):
    if random.random() < pm:
        #swap a random gene with a zone not in chromosome
        size = len(chromosome)
        available = [i for i in range(36) if i not in chromosome]
        if available:
          #find a zone to replace randomly
            idx_to_replace = random.randint(0, size-1)
            chromosome[idx_to_replace] = random.choice(available)
    return chromosome

In [95]:
def tournament_selection(population, k=3):
    competitors = random.sample(population, k)
    return max(competitors, key=lambda chrom: ga_fitness(chrom, demands))

    #pick k chromosomes randomly and choose the fittest

In [105]:
def ga_driver(pop_size, generations, pm):
    #initialize population
    population = [sorted(random_state()) for _ in range(pop_size)]
    best_chromosome = None
    best_fitness_val = -float('inf')

    for gen in range(generations):
        new_population = []

        while len(new_population) < pop_size:
            #selection
            parent1 = tournament_selection(population)
            parent2 = tournament_selection(population)

            #crossover
            child = ordered_crossover(parent1, parent2)

            #mutation
            child = ga_mutate(child, pm)

            new_population.append(child)

        population = new_population

        #keep track of best chromosome
        for chrom in population:
            fit = ga_fitness(chrom, demands)
            if fit > best_fitness_val:
                best_fitness_val = fit
                best_chromosome = chrom

    print("best solution:", best_chromosome)
    print("fitness:", best_fitness_val)
    print("driver zones (row, col):", [(i//6, i%6) for i in best_chromosome])
    return best_chromosome, best_fitness_val, population

In [106]:
ga_driver(pop_size=30, generations=100, pm=0.1)

best solution: [11, 8, 33, 17, 3, 29, 22, 31, 13, 25]
fitness: 700
driver zones (row, col): [(1, 5), (1, 2), (5, 3), (2, 5), (0, 3), (4, 5), (3, 4), (5, 1), (2, 1), (4, 1)]


([11, 8, 33, 17, 3, 29, 22, 31, 13, 25],
 700,
 [[11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 9, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 9],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 14, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 5, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31, 13, 25],
  [11, 8, 33, 3, 22, 17, 29, 31,

In [107]:
bf_rrhc = [] #best fitness for rrhc list
for i in range (20):
  best_state, best_fitness, var = rrhc_driver(num_restarts=30, demands=demands, variant='first_choice')
  bf = bf_rrhc.append(best_fitness)
print (bf_rrhc)

[703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703, 703]


In [113]:
bf_ga = []
for trial in range(20):
    best_chromosome, best_fitness, ret = ga_driver(pop_size=30, generations=100, pm=0.1)
    bf = bf_ga.append(best_fitness)
print("\n",bf_ga)

best solution: [31, 29, 17, 13, 22, 33, 11, 8, 3, 25]
fitness: 700
driver zones (row, col): [(5, 1), (4, 5), (2, 5), (2, 1), (3, 4), (5, 3), (1, 5), (1, 2), (0, 3), (4, 1)]
best solution: [22, 33, 8, 17, 31, 18, 25, 29, 3, 13]
fitness: 703
driver zones (row, col): [(3, 4), (5, 3), (1, 2), (2, 5), (5, 1), (3, 0), (4, 1), (4, 5), (0, 3), (2, 1)]
best solution: [3, 8, 17, 31, 13, 29, 25, 11, 22, 33]
fitness: 700
driver zones (row, col): [(0, 3), (1, 2), (2, 5), (5, 1), (2, 1), (4, 5), (4, 1), (1, 5), (3, 4), (5, 3)]
best solution: [31, 17, 13, 22, 29, 25, 8, 3, 18, 33]
fitness: 703
driver zones (row, col): [(5, 1), (2, 5), (2, 1), (3, 4), (4, 5), (4, 1), (1, 2), (0, 3), (3, 0), (5, 3)]
best solution: [27, 22, 8, 17, 31, 13, 3, 25, 29, 33]
fitness: 697
driver zones (row, col): [(4, 3), (3, 4), (1, 2), (2, 5), (5, 1), (2, 1), (0, 3), (4, 1), (4, 5), (5, 3)]
best solution: [18, 17, 6, 13, 8, 29, 33, 31, 22, 25]
fitness: 692
driver zones (row, col): [(3, 0), (2, 5), (1, 0), (2, 1), (1, 2), (4

In [120]:
#no numpy since not allowed bf_rrhc and best_fitness_ga are lists

#compute mean
def mean(lst):
    return sum(lst)/len(lst)

#compute standard deviation
def std(lst):
    m = mean(lst)
    return (sum((x - m)**2 for x in lst)/len(lst))**0.5

# compute best run
def best(lst):
    return max(lst)

#RRHC stats
mean_rrhc = round(mean(bf_rrhc),2)
std_rrhc = round(std(bf_rrhc),2)
best_rrhc = best(bf_rrhc)

#GA stats
mean_ga = round(mean(best_fitness_ga),2)
std_ga = round(std(best_fitness_ga),2)
best_ga = best(best_fitness_ga)

print("RRHC - mean:", mean_rrhc, "std:", std_rrhc, "best:", best_rrhc)
print("GA - mean:", mean_ga, "std:", std_ga, "best:", best_ga)

RRHC - mean: 703.0 std: 0.0 best: 703
GA - mean: 699.9 std: 4.48 best: 703
